
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Benchmark Evaluation


In this demo, **we will focus on evaluating large language models using a benchmark dataset specific to the task at hand.**

**Learning Objectives:**

*By the end of this demo, you will be able to;*

* Obtain reference/benchmark data set for task-specific LLM evaluation
* Evaluate an LLM's performance on a specific task using task-specific metrics
* Compare relative performance of two LLMs using a benchmark set

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**



## Classroom Setup

Install required libraries.

In [0]:
%pip install databricks-sdk rouge_score evaluate textstat mlflow>=3.0 databricks-feature-engineering --upgrade
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.11.0 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.4 which is incompatible.
langchain 0.1.20 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.4.42 which is incompatible.
langchain 0.1.20 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-community 0.0.38 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.4 which is incompatible.
langchain-community 0.0.38 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.4.42 which is incompatible.
langchain-community 0.0.38 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-text-splitters 0.0.2 requires langchain-core<0.3,

## ROUGE Score: A Complete Tutorial for Evaluating Text Summarization Models

https://medium.com/@prabhatzade/rouge-score-a-complete-tutorial-for-evaluating-text-summarization-models-a3a146417118

https://medium.com/nlplanet/two-minutes-nlp-learn-the-rouge-metric-by-examples-f179cc285499

### 1. ROUGE-1 considering unigrams.
- `ROUGE-1 precision` can be computed as the ratio of the number of unigrams in Generate (G) that appear also in Reference (R) over the number of unigrams in G.
- `ROUGE-1 recall` can be computed as the ratio of the number of unigrams in Reference (R) that appear also in Generate (G) over the number of unigrams in R.
- `ROUGE-1 F1` = ```2 * Recall * Precision /(Recall+Precision)```


### 2. ROUGE-2 considering 2-grams.

- `ROUGE-2 precision` is the ratio of the number of 2-grams in G that appear also in R, over the number of 2-grams in G.

- `ROUGE-2 recall` is the ratio of the number of 2-grams in R that appear also in G, over the number of 2-grams in R.

- `ROUGE-2 F1` = ```2 * Recall * Precision /(Recall+Precision)```



### 3. ROUGE-L considering the longest common subsequence (LCS)

- `ROUGE-L precision` is the ratio of the length of the LCS, over the number of unigrams in G.

- `ROUGE-L precision` is the ratio of the length of the LCS, over the number of unigrams in R.

- `ROUGE-2 F1` = ```2 * Recall * Precision /(Recall+Precision)```


In [0]:
from rouge_score import rouge_scorer

# Sample reference and generated summaries
reference_summary = "The quick brown fox jumps over the lazy dog"
generated_summary = "he quick brown dog jumps on the log."
# ROUGE-1: Score(precision=0.625, recall=0.5555555555555556, fmeasure=0.5882352941176471)
# ROUGE-2: Score(precision=0.14285714285714285, recall=0.125, fmeasure=0.13333333333333333)
# ROUGE-L: Score(precision=0.5, recall=0.4444444444444444, fmeasure=0.47058823529411764)

# reference_summary = "The cat sat on the mat."
# generated_summary = "The cat is on the mat."


'''
#########################################################################################################
# ROUGE-1: Score(precision=0.8333333333333334, recall=0.8333333333333334, fmeasure=0.8333333333333334)
#########################################################################################################
###############
👉 precision:
✅ Unigram overlap = {The, cat, on, the, mat} = 5 matches
✅ Total unigrams in generate = 6
✅ ROUGE-1 Precision = 5/6 = 0.83 (83%)
###############
👉 recall:
✅ Unigram overlap = {The, cat, on, the, mat} = 5 matches
✅ Total unigrams in reference = 6
✅ ROUGE-1 Recall = 5/6 = 0.83 (83%)
###############
👉 fmeasure : 
✅ ROUGE-1 F1 = 2*Recall*Precision /(Recall+Precision) = 2*0.83*0.83/(0.83+0.83) = 0.83
#########################################################################################################



#########################################################################################################
# ROUGE-2: Score(precision=0.6, recall=0.6, fmeasure=0.6)
#########################################################################################################
###############
👉 precision:
✅ 2-gram overlap = {The cat, on the, the mat} = 3 matches
✅ Total 2-grams in generate = 5 ({The cat, cat is, is on, on the, the mat} )
✅ ROUGE-2 Precision = 3/5 = 0.60 (60%)
###############
👉 recall:
✅ 2-gram overlap = {The cat, on the, the mat} = 3 matches
✅ Total 2-grams in reference = 5 ({The cat, cat sat, sat on, on the, the mat} )
✅ ROUGE-2 Recall = 3/5 = 0.60 (60%)
###############
👉 fmeasure : 
✅ ROUGE-2 F1 = 2*Recall*Precision /(Recall+Precision) = 2*0.6*0.6/(0.6+0.6) = 0.6
#########################################################################################################



#########################################################################################################
# ROUGE-L: Score(precision=0.8333333333333334, recall=0.8333333333333334, fmeasure=0.8333333333333334)
#########################################################################################################
The LCS is the 5-gram “the cat on the mat” (remember that the words are not necessarily consecutive), which appears in both R and G
###############
👉 precision:
✅ 5-gram overlap = {The cat sat the mat} = 5 matches
✅ Total unigrams in generate = 6 
✅ ROUGE-L Precision = 5/6 = 0.83 (83%)
###############
👉 recall:
✅ 2-gram overlap = {The cat, on the, the mat} = 3 matches
✅ Total unigrams in reference = 6
✅ ROUGE-L Recall = 5/6 = 0.83 (83%)
###############
👉 fmeasure : 
✅ ROUGE-L F1 = 2*Recall*Precision /(Recall+Precision) = 2*0.83*0.83/(0.83+0.83) = 0.83
#########################################################################################################
'''


# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Compute scores
scores = scorer.score(reference_summary, generated_summary)

# Print results
print("ROUGE-1:", scores['rouge1'])
print("ROUGE-2:", scores['rouge2'])
print("ROUGE-L:", scores['rougeL'])

ROUGE-1: Score(precision=0.625, recall=0.5555555555555556, fmeasure=0.5882352941176471)
ROUGE-2: Score(precision=0.14285714285714285, recall=0.125, fmeasure=0.13333333333333333)
ROUGE-L: Score(precision=0.5, recall=0.4444444444444444, fmeasure=0.47058823529411764)


Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-03


The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12546244_1763064814@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12546244_1763064814
Working Directory: /Volumes/dbacademy/ops/labuser12546244_1763064814@vocareum_com
Dataset Location:  NestedNamespace (news='/Volumes/dbacademy_news/v01', arxiv='/Volumes/dbacademy_arxiv/v01')


## Demo Overview

In this demonstration, we will be evaluating the performance of an AI system designed to summarize text.

The text documents that we will be summarizing are a collection of fictional product reviews for grocery products.

The AI system works as follows:

1. Accepts a text document as input
2. Constructs an LLM prompt using few-shot learning to summarize the text
3. Submits the prompt to an LLM for summarization
4. Returns summarized text

See below for an example of the system.

## Step 1: Setup Models to Use

Next, we will setup the model that will be used for evaluation.

We will use **Databricks Claude 3.7 Sonnet** and **Llama 3.3 70B Instruct** for evaluation.

In [0]:
from databricks.sdk.service.serving import ChatMessage
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Define the first model for summarization
def query_summary_system(input: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that summarizes text. Given a text input, you need to provide a one-sentence summary. You specialize in summarizing reviews of grocery products. Please keep the reviews in first-person perspective if they're originally written in first person. Do not change the sentiment. Do not create a run-on sentence – be concise."
        },
        { 
            "role": "user", 
            "content": input 
        }
    ]
    messages = [ChatMessage.from_dict(message) for message in messages]
    chat_response = w.serving_endpoints.query(
        name="databricks-claude-3-7-sonnet",
        messages=messages,
        temperature=0.1,
        max_tokens=128
    )

    return chat_response.as_dict()["choices"][0]["message"]["content"]

# Define the second model for summarization
def challenger_query_summary_system(input: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that summarizes text. Given a text input, you need to provide a one-sentence summary. You specialize in summarizing reviews of grocery products. Please keep the reviews in first-person perspective if they're originally written in first person. Do not change the sentiment. Do not create a run-on sentence – be concise."
        },
        { 
            "role": "user", 
            "content": input 
        }
    ]
    messages = [ChatMessage.from_dict(message) for message in messages]
    chat_response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        temperature=0.1,
        max_tokens=128
    )

    return chat_response.as_dict()["choices"][0]["message"]["content"]

Let's test the models!

In [0]:
query_summary_system(
    "This is the best frozen pizza I've ever had! Sure, it's not the healthiest, but it tasted just like it was delivered from our favorite pizzeria down the street. The cheese browned nicely and fresh tomatoes are a nice touch, too! I would buy it again despite it's high price. If I could change one thing, I'd made it a little healthier – could we get a gluten-free crust option? My son would love that."
)

"I loved this frozen pizza for its authentic pizzeria taste, nicely browned cheese and fresh tomatoes, though I wish they'd offer a gluten-free option despite the high price."

In [0]:
challenger_query_summary_system(
    "This is the best frozen pizza I've ever had! Sure, it's not the healthiest, but it tasted just like it was delivered from our favorite pizzeria down the street. The cheese browned nicely and fresh tomatoes are a nice touch, too! I would buy it again despite it's high price. If I could change one thing, I'd made it a little healthier – could we get a gluten-free crust option? My son would love that."
)

"I think this is the best frozen pizza I've ever had, with a delicious taste similar to a pizzeria's, and I would buy it again despite its high price."

To complete this workflow, we'll focus on the following steps:

1. Obtain a benchmark set for evaluating summarization
2. Compute summarization-specific evaluation metrics using the benchmark set
3. Compare performance with another LLM using the benchmark set and evaluation metrics

## Step 2: Benchmark and Reference Sets

As a reminder, our task-specific evaluation metrics (including ROUGE for summarization) require a benchmark set to compute scores.

There are two types of reference/benchmark sets that we can use:

1. Large, generic benchmark sets commonly used across use cases
2. Domain-specific benchmark sets specific to your use case

For this demo, we'll focus on the former.

### Generic Benchmark Set

First, we'll import a generic benchmark set used for evaluating text summarization.

We'll use the data set used in [Benchmarking Large Language Models for News Summarization](https://arxiv.org/abs/2301.13848) to evaluate how well our LLM solution summarizes general text.

This data set:

* is relatively large in scale at 599 records
* is related to news articles
* contains original text and *author-written* summaries of the original text

**Question:** What is the advantage of using ground-truth summaries that are written by the original author?

- Ground-truth summaries from the original author provide the gold standard to capture the summary of the article

In [0]:
import pandas as pd

# Read and display the dataset
eval_data = pd.read_csv(f"{DA.paths.datasets.news}/csv/news-summaries.csv")
display(eval_data)

inputs writer_summary Baltimore's mayor has sacked the US city's police chief, saying his leadership had become a distraction from fighting a "crime surge".

Mayor Stephanie Rawlings-Blake said she was replacing Police Commissioner Anthony Batts with his deputy, Kevin Davis, for an interim period.

The city was rocked by riots in April when a black man died after suffering injuries in police custody.

Six officers were charged over the death of the 25-year-old, Freddie Gray.

Speaking at a news conference on Wednesday, Mayor Rawlings-Blake said Mr Batts had "served this city with distinction" since becoming police chief in October 2012.

But referring to the city's high homicide rate, she said "too many continue to die".

"The focus has been too much on the leadership of the department and not enough on the crime fighting," she told reporters, adding: "We need to get the crime surge under control."

The city has seen a sharp increase in violence since Freddie Gray's death on 19 April, with 155 homicides this year, a 48% increase over the same period last year.

On Tuesday, the police department announced that an outside organisation will review its response to the civil unrest that followed Mr Gray's death.

The US justice department is also conducting a civil rights review of the Baltimore force and Mr Batts has been criticised by the city's police union.

Earlier on Wednesday, the union released its report into the police handling of the rioting.

It said officers had complained "that they lacked basic riot equipment, training, and, as events unfolded, direction from leadership".

The report also said "officers repeatedly expressed concern that the passive response to the civil unrest had allowed the disorder to grow into full scale rioting".

Recent events had "placed attention on police leadership", Ms Rawlings-Blake said, but denied her decision was influenced by the union report.

Mr Davis, who is taking over immediately as interim police chief, praised his "friend" Mr Batts and said he was a "true reform commissioner".

Mayor Rawlings-Blake said Mr Davis would "bring accountability to police, hold officers who act out of line accountable for their actions". The mayor of Baltimore fired the police chief and replaced him with his deputy. According to the mayor, crime in the city was unacceptable. Riots in the city after a man died in police custody and a surge in homicide rates were cited as reasons for the firing. Western Sahara has welcomed Morocco's readmission to the African Union, 32 years after members refused to withdraw support for the territory's independence.

It was a "good opportunity" and "a chance to work together," a top Western Sahara official told the BBC.

Morocco controls two-thirds of Western Sahara and sees it as part of its historic territory.

However some, including the UN, see Western Sahara as Africa's last colony.

Africa Live: More on this and other stories

Find out more about Western Sahara

A referendum was promised in 1991 but never carried out due to wrangling over who was eligible to vote.

Thousands of Sahrawi refugees still live in refugee camps in Algeria - some have been there for 40 years.

It is not clear what happens next but Western Sahara is hopeful that a committee set up by the AU will address the issues that both sides have raised.

Some AU delegates said that it would be easier to resolve the issue with Morocco inside the AU.

Sidi Mohammed, a Western Sahara official, told the BBC that Morocco's return to the AU means that it would now be expected to put "in practice decisions taken by the AU with regard to a referendum in Western Sahara".

Mr Mohammed dismissed the suggestion that Morocco would now seek to get the AU to change its position, saying that the no country could unilaterally change the AU fundamental agreement, saying it opposed colonisation.

In his speech at the AU summit, King Mohammed VI of Morocco said the readmission was not meant to divide the continental b

## Step 3: Compute the ROUGE Evaluation Metric

Next, we will want to compute our ROUGE-N metric to understand how well our system summarizes grocery generic text using the benchmark dataset.

We can compute the ROUGE metric (among others) using MLflow's new LLM evaluation capabilities. MLflow LLM evaluation includes default collections of metrics for pre-selected tasks, e.g, “question-answering” or "text-summarization" (our case). Depending on the LLM use case that you are evaluating, these pre-defined collections can greatly simplify the process of running evaluations.

The `mlflow.evaluate` function accepts the following parameters for this use case:

* An LLM model
* Reference data for evaluation (our benchmark set)
* Column with ground truth data
* The model/task type (e.g. `"text-summarization"`)

**Note:** The `text-summarization` type will automatically compute ROUGE-related metrics. For some metrics, additional library installs will be needed – you can see the requirements in the printed output.

In [0]:
# A custom function to iterate through our eval DF
def query_iteration(inputs):
    answers = []

    for index, row in inputs.iterrows():
        completion = query_summary_system(row["inputs"])
        answers.append(completion)

    return answers

# Test query_iteration function – it needs to return a list of output strings
query_iteration(eval_data.head())

["Baltimore's mayor fired the city's police chief, citing his leadership as a distraction from addressing the 48% increase in homicides, and appointed his deputy as interim commissioner to focus on crime fighting rather than departmental leadership.",
 "Western Sahara officials welcomed Morocco's readmission to the African Union after 32 years, viewing it as an opportunity to work together despite ongoing disputes over Western Sahara's independence and Morocco's control of two-thirds of the territory.",
 'England flanker James Haskell shared an Instagram photo of himself dressed as Iron Man to celebrate the release of Avengers: Age of Ultron.',
 'UK manufacturing activity contracted in April for the first time in three years due to soft domestic demand, falling overseas business, and uncertainty ahead of the EU referendum, signaling job losses and acting as a drag on the economy.',
 'A mother-of-six lost over seven stone after being mortified when a child on a bus asked if she was preg

In [0]:
import mlflow

# MLflow's `evaluate` with a custom function
results = mlflow.evaluate(
    query_iteration,                      # iterative function from above
    eval_data.head(50),                   # limiting for speed
    targets="writer_summary",             # column with expected or "good" output
    model_type="text-summarization"       # type of model or task
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-79ab5b16-6168-4be5-b9a0-b6de68b67588/lib/python3.11/site-packages/mlflow/models/evaluation/deprecated.py:9: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(
2025/11/13 20:27:36 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/11/13 20:27:36 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-79ab5b16-6168-4be5-

2025/11/13 20:29:14 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 0.3.7 <= langchain, but the installed version is 0.1.20. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.


We can view the results for individual records by subsetting the handy `.tables` object.

Notice all of the different versions of the ROUGE metric. These are calculated using the HuggingFace `evaluator` library, and the metrics are detailed [here](https://huggingface.co/spaces/evaluate-metric/rouge).

In summary, the descriptions of each metric are below:

* "rouge1": unigram (1-gram) based scoring
* "rouge2": bigram (2-gram) based scoring
* "rougeL": Longest common subsequence based scoring.
* "rougeLSum": splits text using "\n"

In [0]:
display(results.tables["eval_results_table"].head(10))

inputs writer_summary outputs token_count toxicity/v1/score flesch_kincaid_grade_level/v1/score ari_grade_level/v1/score rouge1/v1/score rouge2/v1/score rougeL/v1/score rougeLsum/v1/score Baltimore's mayor has sacked the US city's police chief, saying his leadership had become a distraction from fighting a "crime surge".

Mayor Stephanie Rawlings-Blake said she was replacing Police Commissioner Anthony Batts with his deputy, Kevin Davis, for an interim period.

The city was rocked by riots in April when a black man died after suffering injuries in police custody.

Six officers were charged over the death of the 25-year-old, Freddie Gray.

Speaking at a news conference on Wednesday, Mayor Rawlings-Blake said Mr Batts had "served this city with distinction" since becoming police chief in October 2012.

But referring to the city's high homicide rate, she said "too many continue to die".

"The focus has been too much on the leadership of the department and not enough on the crime fighting," she told reporters, adding: "We need to get the crime surge under control."

The city has seen a sharp increase in violence since Freddie Gray's death on 19 April, with 155 homicides this year, a 48% increase over the same period last year.

On Tuesday, the police department announced that an outside organisation will review its response to the civil unrest that followed Mr Gray's death.

The US justice department is also conducting a civil rights review of the Baltimore force and Mr Batts has been criticised by the city's police union.

Earlier on Wednesday, the union released its report into the police handling of the rioting.

It said officers had complained "that they lacked basic riot equipment, training, and, as events unfolded, direction from leadership".

The report also said "officers repeatedly expressed concern that the passive response to the civil unrest had allowed the disorder to grow into full scale rioting".

Recent events had "placed attention on police leadership", Ms Rawlings-Blake said, but denied her decision was influenced by the union report.

Mr Davis, who is taking over immediately as interim police chief, praised his "friend" Mr Batts and said he was a "true reform commissioner".

Mayor Rawlings-Blake said Mr Davis would "bring accountability to police, hold officers who act out of line accountable for their actions". The mayor of Baltimore fired the police chief and replaced him with his deputy. According to the mayor, crime in the city was unacceptable. Riots in the city after a man died in police custody and a surge in homicide rates were cited as reasons for the firing. Baltimore's mayor fired the city's police chief, citing his leadership as a distraction from addressing the 48% increase in homicides, and appointed his deputy as interim commissioner to focus on crime fighting rather than departmental leadership. 45 1.7943400000000002E-4 21.0666666667 24.3066666667 0.3720930233 0.0952380952 0.2325581395 0.2325581395 Western Sahara has welcomed Morocco's readmission to the African Union, 32 years after members refused to withdraw support for the territory's independence.

It was a "good opportunity" and "a chance to work together," a top Western Sahara official told the BBC.

Morocco controls two-thirds of Western Sahara and sees it as part of its historic territory.

However some, including the UN, see Western Sahara as Africa's last colony.

Africa Live: More on this and other stories

Find out more about Western Sahara

A referendum was promised in 1991 but never carried out due to wrangling over who was eligible to vote.

Thousands of Sahrawi refugees still live in refugee camps in Algeria - some have been there for 40 years.

It is not clear what happens next but Western Sahara is hopeful that a committee set up by the AU will address the issues that both sides have raised.

Some AU delegates said that it would be easier to resolve the issue with Morocco inside the AU.

Sidi Mohammed, a Western Sahara official

And we can view summarized (mean, variance, etc.) model-level (rather than record-level) results with the following:

In [0]:
results.metrics

{'toxicity/v1/mean': 0.00019716921087820083,
 'toxicity/v1/variance': 3.5319447322335464e-08,
 'toxicity/v1/p90': 0.00022813878313172612,
 'toxicity/v1/ratio': 0.0,
 'flesch_kincaid_grade_level/v1/mean': 18.88461729364958,
 'flesch_kincaid_grade_level/v1/variance': 8.970703892492814,
 'flesch_kincaid_grade_level/v1/p90': 22.737000000000005,
 'ari_grade_level/v1/mean': 22.13459047168572,
 'ari_grade_level/v1/variance': 10.422706887375915,
 'ari_grade_level/v1/p90': 25.9226950166113,
 'rouge1/v1/mean': 0.3708690704829302,
 'rouge1/v1/variance': 0.007090418111296535,
 'rouge1/v1/p90': 0.4758064516129033,
 'rouge2/v1/mean': 0.1497278592807707,
 'rouge2/v1/variance': 0.006634913543254246,
 'rouge2/v1/p90': 0.2827935222672065,
 'rougeL/v1/mean': 0.25093829587051536,
 'rougeL/v1/variance': 0.007253762370212212,
 'rougeL/v1/p90': 0.3685344827586207,
 'rougeLsum/v1/mean': 0.25093829587051536,
 'rougeLsum/v1/variance': 0.007253762370212212,
 'rougeLsum/v1/p90': 0.3685344827586207}

In [0]:
results.__dict__.keys()

dict_keys(['_metrics', '_artifacts', '_run_id'])

In [0]:
results.artifacts

{'eval_results_table': JsonEvaluationArtifact(uri='dbfs:/databricks/mlflow-tracking/275941072027999/5c94d60aaea14150baf63f963c9991ba/artifacts/eval_results_table.json')}

In [0]:
results.run_id

'5c94d60aaea14150baf63f963c9991ba'

#### You should use results.tables["eval_results_table"] to access the per-row evaluation results directly, as shown in the MLflow documentation. If you want to display it as a Spark DataFrame, first convert it to a Pandas DataFrame, then to a Spark DataFrame.

In [0]:
import pandas as pd

# Display the per-row evaluation results as a Pandas DataFrame
display(
    pd.DataFrame(
        results.tables["eval_results_table"]
    )
)

inputs writer_summary outputs token_count toxicity/v1/score flesch_kincaid_grade_level/v1/score ari_grade_level/v1/score rouge1/v1/score rouge2/v1/score rougeL/v1/score rougeLsum/v1/score Baltimore's mayor has sacked the US city's police chief, saying his leadership had become a distraction from fighting a "crime surge".

Mayor Stephanie Rawlings-Blake said she was replacing Police Commissioner Anthony Batts with his deputy, Kevin Davis, for an interim period.

The city was rocked by riots in April when a black man died after suffering injuries in police custody.

Six officers were charged over the death of the 25-year-old, Freddie Gray.

Speaking at a news conference on Wednesday, Mayor Rawlings-Blake said Mr Batts had "served this city with distinction" since becoming police chief in October 2012.

But referring to the city's high homicide rate, she said "too many continue to die".

"The focus has been too much on the leadership of the department and not enough on the crime fighting," she told reporters, adding: "We need to get the crime surge under control."

The city has seen a sharp increase in violence since Freddie Gray's death on 19 April, with 155 homicides this year, a 48% increase over the same period last year.

On Tuesday, the police department announced that an outside organisation will review its response to the civil unrest that followed Mr Gray's death.

The US justice department is also conducting a civil rights review of the Baltimore force and Mr Batts has been criticised by the city's police union.

Earlier on Wednesday, the union released its report into the police handling of the rioting.

It said officers had complained "that they lacked basic riot equipment, training, and, as events unfolded, direction from leadership".

The report also said "officers repeatedly expressed concern that the passive response to the civil unrest had allowed the disorder to grow into full scale rioting".

Recent events had "placed attention on police leadership", Ms Rawlings-Blake said, but denied her decision was influenced by the union report.

Mr Davis, who is taking over immediately as interim police chief, praised his "friend" Mr Batts and said he was a "true reform commissioner".

Mayor Rawlings-Blake said Mr Davis would "bring accountability to police, hold officers who act out of line accountable for their actions". The mayor of Baltimore fired the police chief and replaced him with his deputy. According to the mayor, crime in the city was unacceptable. Riots in the city after a man died in police custody and a surge in homicide rates were cited as reasons for the firing. Baltimore's mayor fired the city's police chief, citing his leadership as a distraction from addressing the 48% increase in homicides, and appointed his deputy as interim commissioner to focus on crime fighting rather than departmental leadership. 45 1.7943400000000002E-4 21.0666666667 24.3066666667 0.3720930233 0.0952380952 0.2325581395 0.2325581395 Western Sahara has welcomed Morocco's readmission to the African Union, 32 years after members refused to withdraw support for the territory's independence.

It was a "good opportunity" and "a chance to work together," a top Western Sahara official told the BBC.

Morocco controls two-thirds of Western Sahara and sees it as part of its historic territory.

However some, including the UN, see Western Sahara as Africa's last colony.

Africa Live: More on this and other stories

Find out more about Western Sahara

A referendum was promised in 1991 but never carried out due to wrangling over who was eligible to vote.

Thousands of Sahrawi refugees still live in refugee camps in Algeria - some have been there for 40 years.

It is not clear what happens next but Western Sahara is hopeful that a committee set up by the AU will address the issues that both sides have raised.

Some AU delegates said that it would be easier to resolve the issue with Morocco inside the AU.

Sidi Mohammed, a Western Sahara official

We are also able to review the results in the MLflow Experiment Tracking UI.

### What does good look like?

The ROUGE metrics range between 0 and 1 – where 0 indicates extremely dissimilar text and 1 indicates extremely similar text. However, our interpretation of what is "good" is usually going to be use-case specific. We don't always want a ROUGE score close to 1 because it's likely not reducing the text size too much.

To explore what "good" looks like, let's review a couple of our examples.

In [0]:
import pandas as pd
display(
    pd.DataFrame(
        results.tables["eval_results_table"]
    ).loc[0:1, ["inputs", "outputs", "rouge1/v1/score"]]
)

inputs,outputs,rouge1/v1/score
"Baltimore's mayor has sacked the US city's police chief, saying his leadership had become a distraction from fighting a ""crime surge"". Mayor Stephanie Rawlings-Blake said she was replacing Police Commissioner Anthony Batts with his deputy, Kevin Davis, for an interim period. The city was rocked by riots in April when a black man died after suffering injuries in police custody. Six officers were charged over the death of the 25-year-old, Freddie Gray. Speaking at a news conference on Wednesday, Mayor Rawlings-Blake said Mr Batts had ""served this city with distinction"" since becoming police chief in October 2012. But referring to the city's high homicide rate, she said ""too many continue to die"". ""The focus has been too much on the leadership of the department and not enough on the crime fighting,"" she told reporters, adding: ""We need to get the crime surge under control."" The city has seen a sharp increase in violence since Freddie Gray's death on 19 April, with 155 homicides this year, a 48% increase over the same period last year. On Tuesday, the police department announced that an outside organisation will review its response to the civil unrest that followed Mr Gray's death. The US justice department is also conducting a civil rights review of the Baltimore force and Mr Batts has been criticised by the city's police union. Earlier on Wednesday, the union released its report into the police handling of the rioting. It said officers had complained ""that they lacked basic riot equipment, training, and, as events unfolded, direction from leadership"". The report also said ""officers repeatedly expressed concern that the passive response to the civil unrest had allowed the disorder to grow into full scale rioting"". Recent events had ""placed attention on police leadership"", Ms Rawlings-Blake said, but denied her decision was influenced by the union report. Mr Davis, who is taking over immediately as interim police chief, praised his ""friend"" Mr Batts and said he was a ""true reform commissioner"". Mayor Rawlings-Blake said Mr Davis would ""bring accountability to police, hold officers who act out of line accountable for their actions"".","Baltimore's mayor fired the city's police chief, citing his leadership as a distraction from addressing the 48% increase in homicides, and appointed his deputy as interim commissioner to focus on crime fighting rather than departmental leadership.",0.3720930233
"Western Sahara has welcomed Morocco's readmission to the African Union, 32 years after members refused to withdraw support for the territory's independence. It was a ""good opportunity"" and ""a chance to work together,"" a top Western Sahara official told the BBC. Morocco controls two-thirds of Western Sahara and sees it as part of its historic territory. However some, including the UN, see Western Sahara as Africa's last colony. Africa Live: More on this and other stories Find out more about Western Sahara A referendum was promised in 1991 but never carried out due to wrangling over who was eligible to vote. Thousands of Sahrawi refugees still live in refugee camps in Algeria - some have been there for 40 years. It is not clear what happens next but Western Sahara is hopeful that a committee set up by the AU will address the issues that both sides have raised. Some AU delegates said that it would be easier to resolve the issue with Morocco inside the AU. Sidi Mohammed, a Western Sahara official, told the BBC that Morocco's return to the AU means that it would now be expected to put ""in practice decisions taken by the AU with regard to a referendum in Western Sahara"". Mr Mohammed dismissed the suggestion that Morocco would now seek to get the AU to change its position, saying that the no country could unilaterally change the AU fundamental agreement, saying it opposed colonisation. In his speech at the AU summit, King Mohammed VI of Morocco said the readmission was not meant to divide th

**Discussion Questions:**
1. How do you interpret the ROUGE-1 score?

- ROUGE-1 considering unigrams.
  - `ROUGE-1 precision` can be computed as the ratio of the number of unigrams in Generate (G) that appear also in Reference (R) over the number of unigrams in G.
  - `ROUGE-1 recall` can be computed as the ratio of the number of unigrams in Reference (R) that appear also in Generate (G) over the number of unigrams in R.
  - `ROUGE-1 F1` = ```2 * Recall * Precision /(Recall+Precision)```
  
2. Do the scores reflect the summarization that you think is best?

- score does not provide the best explanation for the task but it provides some baselines to evaluate the performance of model

## Step 4: Comparing LLM Performance

In practice, we will frequently be comparing LLMs (or larger AI systems) against one another when determining which is the best for our use case. As a result of this, it's important to become familiar with comparing these solutions.

In the below cell, we demonstrate computing the same metrics using the same reference dataset – but this time, we're summarizing using a system that utilizes a different LLM.

**Note:** This time, we're going to read our reference dataset from Delta.

In [0]:
# A compare custom function to iterate through our eval DF
def challenger_query_iteration(inputs):
    answers = []

    for index, row in inputs.iterrows():
        completion = challenger_query_summary_system(row["inputs"])
        answers.append(completion)

    return answers

# Compute challenger results
challenger_results = mlflow.evaluate(
    challenger_query_iteration,           # iterative function from above
    eval_data.head(50),
    targets="writer_summary",             # column with expected or "good" output
    model_type="text-summarization"       # type of model or task
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-79ab5b16-6168-4be5-b9a0-b6de68b67588/lib/python3.11/site-packages/mlflow/models/evaluation/deprecated.py:9: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(
2025/11/13 20:29:19 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-79ab5b16-6168-4be5-b9a0-b6de68b67588/lib/python3.11/site-packages/mlflow/models/evaluation/evaluators/default.py:100: FutureWarning: ``mlflow.metrics.token_count`` is deprecated since 3.4.0. Use the new GenAI evaluation functionality instead. Se

Let's take a look at our model-level results.

In [0]:
challenger_results.metrics

{'toxicity/v1/mean': 0.0003315153621952049,
 'toxicity/v1/variance': 4.484500670419427e-07,
 'toxicity/v1/p90': 0.00029352591081988077,
 'toxicity/v1/ratio': 0.0,
 'flesch_kincaid_grade_level/v1/mean': 17.40302473132635,
 'flesch_kincaid_grade_level/v1/variance': 4.694915376594951,
 'flesch_kincaid_grade_level/v1/p90': 20.083333333333332,
 'ari_grade_level/v1/mean': 19.851149142634373,
 'ari_grade_level/v1/variance': 10.922834963748393,
 'ari_grade_level/v1/p90': 23.461116322701695,
 'rouge1/v1/mean': 0.3182489512001712,
 'rouge1/v1/variance': 0.007452455665202287,
 'rouge1/v1/p90': 0.4348940914158305,
 'rouge2/v1/mean': 0.10757253965892179,
 'rouge2/v1/variance': 0.0054627915047626985,
 'rouge2/v1/p90': 0.17047543997898607,
 'rougeL/v1/mean': 0.20993921180892447,
 'rougeL/v1/variance': 0.005824878709469913,
 'rougeL/v1/p90': 0.29197368421052633,
 'rougeLsum/v1/mean': 0.20993921180892447,
 'rougeLsum/v1/variance': 0.005824878709469913,
 'rougeLsum/v1/p90': 0.29197368421052633}

And let's compare in the MLflow UI, looking at the experiment's **Chart** tab.

**Note:** We can filter specifically to ROUGE metrics.

  

LINK : https://dbc-7df47a97-2508.cloud.databricks.com/ml/compare-runs?runs=%5B%2223c17287e68e4be6aadf4048b5c943ed%22%2C%225c94d60aaea14150baf63f963c9991ba%22%5D&experiments=%5B%22275941072027999%22%5D&o=1498738153398017



| Metric               | databricks-meta-llama-3-3-70b-instruct | databricks-claude-3-7-sonnet |
|----------------------|---------|---------|
| rouge1/v1/mean       | 0.318   | 0.371   |
| rouge1/v1/p90        | 0.435   | 0.476   |
| rouge1/v1/variance   | 0.007   | 0.007   |
| rouge2/v1/mean       | 0.108   | 0.15    |
| rouge2/v1/p90        | 0.17    | 0.283   |
| rouge2/v1/variance   | 0.005   | 0.007   |
| rougeL/v1/mean       | 0.21    | 0.251   |
| rougeL/v1/p90        | 0.292   | 0.369   |
| rougeL/v1/variance   | 0.006   | 0.007   |
| rougeLsum/v1/mean    | 0.21    | 0.251   |
| rougeLsum/v1/p90     | 0.292   | 0.369   |
| rougeLsum/v1/variance| 0.006   | 0.007   |


### What about other tasks/metrics?

The `mlflow` library contains [a number of LLM task evaluation tools](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.evaluate) that we can use in our workflows.


## Conclusion

You should now be able to:

* Obtain reference/benchmark data set for task-specific LLM evaluation
* Evaluate an LLM's performance on a specific task using task-specific metrics
* Compare relative performance of two LLMs using a benchmark set

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>